# Introduction to DuckDB

DuckDB is an in-process, analytical SQL database. You can think of it as SQLite for analytics (OLAP), not transactions (OLTP).

Instead of running as a server, DuckDB runs inside your application process - Python, R, Java, C++, etc. and queries data where it already lives.

## What makes DuckDB special

#### Built for analytics

DuckDB is optimized for:

- Large scans
- Aggregations
- Joins
- Window functions
- Complex analytical queries

Think: dashboards, feature engineering, ad-hoc analysis, data science---not user logins or high-frequency writes.

#### In-process (no server)

There's:

- No daemon
- No port
- No client/server overhead

You just:

```python
import duckdb
duckdb.sql("SELECT * FROM 'data.parquet'")
```

#### Columnar + vectorized execution

Under the hood, DuckDB:

- Stores data in columnar format
- Processes data in vectors (chunks) instead of row-by-row

This is why it's fast. Very fast. Often competitive with Spark for single-machine workloads.

#### Reads data directly (zero-copy vibes)

DuckDB can query:

- Parquet
- CSV
- JSON
- Arrow
- Pandas / Polars DataFrames
- S3 / HTTP / local files

Without importing them first.

```sql
SELECT avg(price)
FROM read_parquet('s3://bucket/sales/*.parquet');
```


### How it compares to other tools

| Tool     | Best at                  | DuckDB difference                                        |
| -------- | ------------------------ | -------------------------------------------------------- |
| SQLite   | App storage              | DuckDB is analytical                                     |
| Postgres | Transactions + analytics | DuckDB is faster for scans, simpler to embed             |
| Pandas   | In-memory analysis       | DuckDB scales better + SQL                               |
| Spark    | Distributed big data     | DuckDB is single-node but simpler & often faster locally |
| BigQuery | Cloud analytics          | DuckDB is local, offline, cheap (free)                   |


### Typical use cases

DuckDB shines when you:

- Do data analysis in Python/R
- Work with Parquet files
- Need SQL over DataFrames
- Want reproducible, local analytics
- Don't want to run a database server

Examples:

- Feature engineering for ML
- Exploratory data analysis
- Replacing slow pandas groupbys
- Local analytics before pushing to a warehouse


### Limitations

DuckDB is not:

- A transactional database
- Multi-user concurrent server
- Meant for high-write workloads

Rule of thumb:

> If you'd normally reach for Spark, BigQuery, or heavy pandas, reach for DuckDB
> If you need users, writes, and locks, consider Postgres


## Installing DuckDB

If you set up the course environment from the repository root with `python -m pip install -r requirements.txt`, DuckDB is already installed.

Otherwise, if you're using Python without conda, install with pip:

```bash
pip install duckdb
```

If you're using a conda environment, install with:

```bash
conda install python-duckdb -c conda-forge
```

:::{tip} What is conda-forge?

conda-forge is a community-run package repository for Conda. It provides pre-built packages for a wide range of software, including DuckDB, making it easy to install and manage dependencies in Conda environments.

**Why conda-forge exists**

The default Conda channel (`defaults`, historically from Anaconda Inc.) has:

- Slower updates
- Fewer packages
- Occasional version lag or mismatches

conda-forge was created to:

- Provide newer packages faster
- Support more platforms
- Ensure everything works together

It's run by hundreds of open-source contributors.

:::


## Using DuckDB

Using DuckDB is as simple as importing the library and running SQL queries. Here's a quick example:


In [1]:
import duckdb

duckdb.sql("SELECT 1").show()

┌───────┐
│   1   │
│ int32 │
├───────┤
│     1 │
└───────┘



### Reading data with DuckDB

DuckDB can read data from multiple sources, including Parquet files, CSVs, and even directly from Pandas DataFrames. You can query these data sources using SQL syntax, making it easy to perform complex analyses without needing to load everything into memory first.

From [DuckDB Documentation](https://duckdb.org/docs/stable/clients/python/overview):

```python
import duckdb

duckdb.read_csv("example.csv")                # read a CSV file into a Relation
duckdb.read_parquet("example.parquet")        # read a Parquet file into a Relation
duckdb.read_json("example.json")              # read a JSON file into a Relation

duckdb.sql("SELECT * FROM 'example.csv'")     # directly query a CSV file
duckdb.sql("SELECT * FROM 'example.parquet'") # directly query a Parquet file
duckdb.sql("SELECT * FROM 'example.json'")    # directly query a JSON file
```


DuckDB can query a pandas DataFrame directly by its variable name. The DataFrame is already in memory, and DuckDB reads it in place, without copying it into a database table first. This allows you to leverage the power of SQL for data manipulation while still working with familiar DataFrame structures.

In [2]:
import pandas as pd

pandas_df = pd.DataFrame({"name": ["John", "Jane"], "age": [30, 25]})
duckdb.sql("SELECT * FROM pandas_df")

nameVARCHAR,ageBIGINT
John,30
Jane,25


### Persistent Storage

You can create a connection to a persistent database using `duckdb.connect(dbname)`, although DuckDB is often used for ad-hoc analysis without needing to manage a database file.

For our credit card transactions analysis, we'll be using DuckDB to query Parquet files directly, so we won't need to set up a persistent database.


## More Examples

Below are a few additional DuckDB patterns you'll see throughout the analysis
notebook.


### Aggregations with `GROUP BY`

DuckDB supports a large part of standard SQL, with a dialect close to PostgreSQL's,
so most `GROUP BY` queries you'd run in PostgreSQL work the same way.

In [3]:
sales = pd.DataFrame(
    {
        "region": ["NA", "EU", "NA", "EU", "APAC", "APAC", "NA"],
        "product": ["A", "A", "B", "B", "A", "B", "B"],
        "units_sold": [120, 80, 45, 60, 90, 30, 70],
        "revenue": [12000.0, 8500.0, 6750.0, 9000.0, 9300.0, 4500.0, 10500.0],
    }
)

duckdb.sql(
    """
SELECT
    region,
    SUM(units_sold)            AS total_units,
    ROUND(SUM(revenue), 2)     AS total_revenue,
    ROUND(AVG(revenue), 2)     AS avg_revenue,
    COUNT(*)                   AS n_orders
FROM sales
GROUP BY region
ORDER BY total_revenue DESC
"""
)

regionVARCHAR,total_unitsHUGEINT,total_revenueDOUBLE,avg_revenueDOUBLE,n_ordersBIGINT
NA,235,29250.0,9750.0,3
EU,140,17500.0,8750.0,2
APAC,120,13800.0,6900.0,2


### Window Functions

Window functions are DuckDB's superpower for transaction data. The query below
ranks each product within its region and computes a running total of revenue.


In [4]:
duckdb.sql(
    """
SELECT
    region,
    product,
    revenue,
    SUM(revenue) OVER (PARTITION BY region ORDER BY product
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                      ) AS running_revenue,
    RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS region_rank
FROM sales
ORDER BY region, region_rank
"""
)

regionVARCHAR,productVARCHAR,revenueDOUBLE,running_revenueDOUBLE,region_rankBIGINT
APAC,A,9300.0,9300.0,1
APAC,B,4500.0,13800.0,2
EU,B,9000.0,17500.0,1
EU,A,8500.0,8500.0,2
NA,A,12000.0,12000.0,1
NA,B,10500.0,29250.0,2
NA,B,6750.0,18750.0,3


:::{tip} `PARTITION BY` is the SQL equivalent of `GROUP BY` for window functions
Where `GROUP BY` collapses rows, `PARTITION BY` keeps every row but evaluates
the window function within each group. We'll use this pattern to compute
**rolling spend per user** in the analysis notebook.
:::


### Reading Parquet Files Directly

DuckDB can query a Parquet file (local or remote) without loading the whole
file into memory. The query planner pushes filters and column selections down
to the file, so only the columns and row groups you need are read.

For a remote file over HTTPS, DuckDB uses its `httpfs` extension. DuckDB
usually installs and loads it automatically the first time you query a URL,
but you can also do it yourself with `INSTALL httpfs` and `LOAD httpfs`. The
example below reads a 30 MB Parquet file of data job postings from the course
datasets repository (the same data as the Data Job Postings case study). It
needs internet access. Because the query uses only two columns, DuckDB
downloads only the parts of the file that hold those columns.

In [5]:
# Query a remote Parquet file by URL (no download step required).
duckdb.sql("INSTALL httpfs")
duckdb.sql("LOAD httpfs")

remote_url = (
    "https://media.githubusercontent.com/media/bdi593/datasets/main/"
    "data-jobs/data_jobs.parquet"
)

duckdb.sql(
    f"""
    SELECT
        job_title_short,
        COUNT(*)                          AS n_postings,
        ROUND(MEDIAN(salary_year_avg), 0) AS median_salary
    FROM read_parquet('{remote_url}')
    GROUP BY job_title_short
    ORDER BY n_postings DESC
    """
).show()

┌───────────────────────────┬────────────┬───────────────┐
│      job_title_short      │ n_postings │ median_salary │
│          varchar          │   int64    │    double     │
├───────────────────────────┼────────────┼───────────────┤
│ Data Analyst              │     196075 │       90000.0 │
│ Data Engineer             │     186241 │      125000.0 │
│ Data Scientist            │     172286 │      127500.0 │
│ Business Analyst          │      49063 │       85000.0 │
│ Software Engineer         │      44929 │       99150.0 │
│ Senior Data Engineer      │      44563 │      147500.0 │
│ Senior Data Scientist     │      36957 │      155500.0 │
│ Senior Data Analyst       │      29216 │      111175.0 │
│ Machine Learning Engineer │      14080 │      106415.0 │
│ Cloud Engineer            │      12331 │       90000.0 │
└───────────────────────────┴────────────┴───────────────┘
  10 rows                                      3 columns



### Mixing SQL and DataFrames Freely

DuckDB's tightest integration with Python is its ability to query any pandas
or Polars DataFrame in scope, and to return results back as either format with
`.df()` or `.pl()`.


In [6]:
# .df()  -> pandas DataFrame
result_pd = duckdb.sql(
    "SELECT region, SUM(revenue) AS total FROM sales GROUP BY region"
).df()
print(type(result_pd).__name__, result_pd.shape)
result_pd

DataFrame (3, 2)


,region,total
0,NA,29250.0
1,EU,17500.0
2,APAC,13800.0


:::{important} When to choose DuckDB over Polars
Use **DuckDB** when:

- You and your team think in SQL.
- You need to query Parquet files that don't fit in memory.
- You want the query planner to handle filter / projection pushdown for free.

Use **Polars** when:

- You want a fluent, chainable Python API.
- The transformation is more _algorithmic_ than _set-based_.
- You're integrating with other Python objects (NumPy arrays, ML libraries).

In practice, many teams use both: Polars for the data-engineering layer,
DuckDB for ad-hoc analytical queries.
:::


## Summary

In this notebook you:

- Saw where DuckDB fits: an in-process analytical SQL engine, not a transactional database server.
- Ran SQL directly on pandas DataFrames, including `GROUP BY` aggregations and window functions with `PARTITION BY`.
- Queried a remote Parquet file by URL, reading only the columns the query needs.
- Converted query results back to pandas with `.df()` (or Polars with `.pl()`).

The analysis notebook applies these patterns to 24 million credit card transactions.